In [ ]:
# Install basic requirements
%pip install librosa
%pip install pystoi
%pip install soundfile==0.12.0
%pip install torch==2.6.0
%pip install torchaudio==2.6.0
%pip install soxr

In [ ]:
import os
# Setup the path to import modules
import sys

repo_root = os.path.abspath("src")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

In [ ]:
# Load in some audio
import soundfile as sf
import soxr

model_fs = 16000

clip_start = 8200000
clip_end = 8400000

aria_audio, fs = sf.read("data/chime9_echi/aria/dev/dev_10.aria.wav")

aria_audio = aria_audio[clip_start:clip_end]
aria_audio = soxr.resample(aria_audio, fs, model_fs)

rainbow_audio, fs = sf.read("data/chime9_echi/participant/dev/P179.wav")
rainbow_audio = soxr.resample(rainbow_audio, fs, model_fs)

In [ ]:
# Play back a noisy sample
from IPython.display import Audio

Audio(aria_audio.T, rate=model_fs)

In [ ]:
from omegaconf import OmegaConf

from shared.core_utils import get_model
from shared.signal_utils import STFTWrapper

cfg = OmegaConf.load("checkpoints/aria_config.yaml")

stft = STFTWrapper(**cfg.input.stft)
model = get_model(cfg, "checkpoints/aria_baseline.pt")  # type: ignore

In [ ]:
import torch

aria_audio = torch.from_numpy(aria_audio.T)
rainbow_audio = torch.from_numpy(rainbow_audio).unsqueeze(0)

In [ ]:
aria_stft = stft(aria_audio).unsqueeze(0).to(torch.float)
rainbow_stft = stft(rainbow_audio).to(torch.float)
rainbow_len = torch.tensor([rainbow_stft.shape[2]]).unsqueeze(0)

output_stft = model(aria_stft, rainbow_stft, rainbow_len).squeeze(1)

In [ ]:
output_audio = stft.inverse(output_stft).squeeze(1).detach().numpy()

Audio(output_audio, rate=model_fs)